In [2]:
import os 

from flowers.evalutation.metrics import prepare_data, evaluate_embedding_regressor, evaluate_embedding_classifier
from sklearn.neural_network import MLPClassifier, MLPRegressor
from sklearn.linear_model import LinearRegression, LogisticRegression
from dotenv import load_dotenv
from datasets import load_dataset

load_dotenv() 

DATA_ROOT = os.getenv("DATA_ROOT")
RANDOM_STATE = 42

/home/phys2526/WhatWeDontCatalog/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [9]:
# Note: REPLACE experiment name and model number for your specfic training run. 

data_files={
    "train": f"{DATA_ROOT}/rgbmnist/rgbmnist_Flow_cond_prior/embeddings/7518770_0/train/*.parquet",
    "test": f"{DATA_ROOT}/rgbmnist/rgbmnist_Flow_cond_prior/embeddings/7518770_0/test/*.parquet"
}
ds = load_dataset(
    "parquet",
    data_files=data_files
)
print(ds)

DatasetDict({
    train: Dataset({
        features: ['orig', 'cond', 'uncond', 'y', 'r', 'g', 'b', 'digit'],
        num_rows: 48000
    })
    test: Dataset({
        features: ['orig', 'cond', 'uncond', 'y', 'r', 'g', 'b', 'digit'],
        num_rows: 10000
    })
})


In [10]:
embed_types = ['orig', 'cond', 'uncond']
features = ["digit", "r", "g", "b"]

In [11]:
clf_model = MLPClassifier(
    hidden_layer_sizes=(64, 32), 
    max_iter=1000, 
    random_state=42
)
reg_model = MLPRegressor(
    hidden_layer_sizes=(64, 32), 
    max_iter=1000, 
    random_state=42
)

In [7]:
clf_models = {
    'log_regression': LogisticRegression(
        random_state=RANDOM_STATE,
        max_iter=1000
    ),
    '2-mlp': MLPClassifier(
        hidden_layer_sizes=(64, 32), 
        max_iter=1000, 
        random_state=RANDOM_STATE
    )
}

reg_models = {
    'lin_regression': LinearRegression(),
    '2-mlp': MLPRegressor(
        hidden_layer_sizes=(64, 32), 
        max_iter=1000, 
        random_state=42
    )
}

In [14]:
all_results = []

for embed_type in embed_types:
    print(embed_type)
    for feature in features:
        print(feature)

        X_train, y_train, X_test, y_test = prepare_data(
            ds=ds,
            embed_type=embed_type,
            factor=feature,
        )
        if feature == "digit":
            for clf_model in clf_models.values():
                evaluate_embedding_classifier(
                    X_train=X_train,
                    X_test=X_test,
                    y_train=y_train,
                    y_test=y_test,
                    model=clf_model
                )
        else:
            for reg_model in reg_models.values():
                evaluate_embedding_regressor(
                    X_train=X_train,
                    X_test=X_test,
                    y_train=y_train,
                    y_test=y_test,
                    model=reg_model
                )
    print("-----")

orig
digit
--- Classification Results ---
Test F1: 0.9470
Test Accuracy: 0.9471

Bootstrap F1 Mean: 0.9470
Bootstrap F1 Median: 0.9471
   95% CI: [0.9426, 0.9514]
   68% CI: [0.9447, 0.9493]
   Err 95: [0.0044]

Bootstrap Accuracy Mean: 0.9471
Bootstrap Accuracy Median: 0.9472
   95% CI: [0.9427, 0.9515]
   68% CI: [0.9448, 0.9494]
   Err 95: [0.0044]
----------------------------------------
--- Classification Results ---
Test F1: 0.9772
Test Accuracy: 0.9772

Bootstrap F1 Mean: 0.9772
Bootstrap F1 Median: 0.9773
   95% CI: [0.9742, 0.9802]
   68% CI: [0.9756, 0.9787]
   Err 95: [0.0030]

Bootstrap Accuracy Mean: 0.9772
Bootstrap Accuracy Median: 0.9773
   95% CI: [0.9742, 0.9802]
   68% CI: [0.9756, 0.9787]
   Err 95: [0.0030]
----------------------------------------
r
--- Regression Results ---
Test R2: 0.9605
Bootstrap R2 Mean: 0.9605
Bootstrap R2 Median: 0.9605
   95% CI: [0.9590, 0.9619]
   68% CI: [0.9597, 0.9613]
   Err 95: [0.0014]
----------------------------------------
--- R